# 03 - Feature Engineering pour Clustering

## 🎯 Objectif
Créer des **profils utilisateurs** pour la segmentation K-Means

## 📋 Stratégie
À partir des fichiers cleaned (`recipes_clean.csv` + `interactions_clean.csv`) :
1. **Joindre interactions ↔ recettes**
2. **Agréger par utilisateur** pour créer des features :
   - Comportement (n_interactions, avg_rating)
   - Préférences temporelles (avg_minutes, avg_n_steps, etc.)
   - Profil nutritionnel (avg_calories, avg_protein, etc.)
   - Tags favoris (proportion de chaque catégorie)
3. **Filtrer utilisateurs** avec trop peu d'interactions (bruit)
4. **Export final** : `users_profiles.csv`

## 📦 Output Final :
- **`users_profiles.csv`** : 1 ligne = 1 utilisateur avec ~21 features pour clustering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_style("whitegrid")
DATA_PROCESSED = Path('../data/processed')

print('✅ Configuration OK')
print(f'   📂 Data processed: {DATA_PROCESSED}')

## ÉTAPE 1 - Charger les Fichiers Cleaned

In [ ]:
print("=" * 80)
print("CHARGEMENT DES FICHIERS CLEANED")
print("=" * 80)

# Charger les fichiers nettoyés
recipes_clean = pd.read_csv(DATA_PROCESSED / 'recipes_clean.csv')
interactions_clean = pd.read_csv(DATA_PROCESSED / 'interactions_clean.csv')

print(f'\n📊 DONNÉES CHARGÉES:')
print(f'   • Recipes clean:      {recipes_clean.shape}')
print(f'   • Interactions clean: {interactions_clean.shape}')

print(f'\n👥 UTILISATEURS UNIQUES:')
print(f'   • Dans interactions: {interactions_clean["user_id"].nunique():,}')

print(f'\n🍳 RECETTES UNIQUES:')
print(f'   • Dans interactions: {interactions_clean["recipe_id"].nunique():,}')
print(f'   • Dans recipes:      {recipes_clean["id"].nunique():,}')

print(f'\n📋 COLONNES RECIPES:')
print(f'   {recipes_clean.columns.tolist()}')

print(f'\n📋 COLONNES INTERACTIONS:')
print(f'   {interactions_clean.columns.tolist()}')

## ÉTAPE 2 - Joindre Interactions ↔ Recettes

In [ ]:
print("=" * 80)
print("JOINTURE INTERACTIONS ↔ RECETTES")
print("=" * 80)

# Colonnes nutrition et tags
nutrition_cols = ['calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates']
tag_cols = ['tag_time_to_make', 'tag_course', 'tag_main_ingredient', 'tag_dietary', 'tag_easy', 'tag_occasion', 'tag_cuisine']

# Sélectionner les colonnes nécessaires des recettes
recipes_cols = ['id', 'minutes', 'n_steps', 'n_ingredients'] + nutrition_cols + tag_cols

# Jointure
interactions_enriched = interactions_clean.merge(
    recipes_clean[recipes_cols],
    left_on='recipe_id',
    right_on='id',
    how='inner'  # inner join pour ne garder que les matchs
)

print(f'\n📊 RÉSULTAT JOINTURE:')
print(f'   • Interactions avant:  {len(interactions_clean):,}')
print(f'   • Interactions après:  {len(interactions_enriched):,}')
print(f'   • Perdues (no match):  {len(interactions_clean) - len(interactions_enriched):,}')

print(f'\n✅ Interactions enrichies: {interactions_enriched.shape}')
print(f'   • Nouvelles colonnes: {len(nutrition_cols) + len(tag_cols) + 3} (nutrition + tags + recipe features)')

## ÉTAPE 3 - Créer les Profils Utilisateurs (Agrégation)

In [ ]:
print("=" * 80)
print("AGRÉGATION PAR UTILISATEUR - CRÉATION DES PROFILS CULINAIRES")
print("=" * 80)

# Agréger par user_id
user_profiles = interactions_enriched.groupby('user_id').agg({
    # Comportement
    'recipe_id': 'count',  # nombre d'interactions
    'rating': 'mean',      # rating moyen
    
    # Temps de cuisson & complexité
    'minutes': 'mean',     # temps moyen
    'n_steps': 'mean',     # complexité moyenne
    'n_ingredients': 'mean',  # nombre moyen d'ingrédients
    
    # Profil nutritionnel (moyenne)
    'calories': 'mean',
    'total_fat': 'mean',
    'sugar': 'mean',
    'sodium': 'mean',
    'protein': 'mean',
    'saturated_fat': 'mean',
    'carbohydrates': 'mean',
    
    # Tags favoris (proportion)
    'tag_time_to_make': 'mean',
    'tag_course': 'mean',
    'tag_main_ingredient': 'mean',
    'tag_dietary': 'mean',
    'tag_easy': 'mean',
    'tag_occasion': 'mean',
    'tag_cuisine': 'mean',
}).reset_index()

# Renommer les colonnes pour plus de clarté
user_profiles.columns = [
    'user_id', 
    'n_interactions', 
    'avg_rating',
    'avg_minutes', 
    'avg_n_steps', 
    'avg_n_ingredients',
    'avg_calories', 
    'avg_total_fat', 
    'avg_sugar', 
    'avg_sodium', 
    'avg_protein', 
    'avg_saturated_fat', 
    'avg_carbohydrates',
    'pref_tag_time_to_make',
    'pref_tag_course',
    'pref_tag_main_ingredient',
    'pref_tag_dietary',
    'pref_tag_easy',
    'pref_tag_occasion',
    'pref_tag_cuisine'
]

print(f'\n✅ PROFILS UTILISATEURS CRÉÉS:')
print(f'   • Nombre d\'utilisateurs: {len(user_profiles):,}')
print(f'   • Nombre de features: {len(user_profiles.columns) - 1}')  # -1 pour user_id
print(f'   • Shape: {user_profiles.shape}')

print(f'\n📊 APERÇU DES PROFILS:')
print(user_profiles.head(10))

print(f'\n📈 STATISTIQUES GLOBALES:')
print(user_profiles.describe())

## ÉTAPE 4 - Filtrer les Utilisateurs (min interactions)

In [ ]:
print("=" * 80)
print("FILTRAGE DES UTILISATEURS")
print("=" * 80)

# Filtrer les utilisateurs avec trop peu d'interactions (bruit pour le clustering)
min_interactions = 5  # Au moins 5 interactions pour avoir un profil significatif

print(f'\n🧹 FILTRAGE (min {min_interactions} interactions):')
print(f'   Avant: {len(user_profiles):,}')

user_profiles_filtered = user_profiles[user_profiles['n_interactions'] >= min_interactions].copy()

print(f'   Après: {len(user_profiles_filtered):,}')
print(f'   Supprimés: {len(user_profiles) - len(user_profiles_filtered):,} ({100*(len(user_profiles)-len(user_profiles_filtered))/len(user_profiles):.1f}%)')

# Vérifier les valeurs manquantes
print(f'\n📊 VALEURS MANQUANTES:')
missing = user_profiles_filtered.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print('   ✅ Aucune valeur manquante')

# Statistiques finales
print(f'\n📈 DISTRIBUTION DES INTERACTIONS PAR UTILISATEUR:')
print(user_profiles_filtered['n_interactions'].describe())

print(f'\n✅ PROFILS UTILISATEURS FINAUX:')
print(f'   • Utilisateurs: {len(user_profiles_filtered):,}')
print(f'   • Features: {len(user_profiles_filtered.columns) - 1}')
print(f'   • Shape: {user_profiles_filtered.shape}')

## ÉTAPE 5 - Visualisations des Profils

In [ ]:
print("=" * 80)
print("VISUALISATIONS DES PROFILS UTILISATEURS")
print("=" * 80)

# Distribution des variables clés
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Interactions
user_profiles_filtered['n_interactions'].hist(bins=50, ax=axes[0, 0], color='steelblue', edgecolor='black')
axes[0, 0].set_title('Distribution - Nombre d\'interactions', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Nombre d\'interactions')
axes[0, 0].set_ylabel('Nombre d\'utilisateurs')

# Rating moyen
user_profiles_filtered['avg_rating'].hist(bins=30, ax=axes[0, 1], color='coral', edgecolor='black')
axes[0, 1].set_title('Distribution - Rating moyen', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Rating moyen')
axes[0, 1].set_ylabel('Nombre d\'utilisateurs')

# Temps moyen
user_profiles_filtered['avg_minutes'].hist(bins=50, ax=axes[0, 2], color='green', edgecolor='black')
axes[0, 2].set_title('Distribution - Temps de cuisson moyen', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Minutes moyennes')
axes[0, 2].set_ylabel('Nombre d\'utilisateurs')

# Calories moyennes
user_profiles_filtered['avg_calories'].hist(bins=50, ax=axes[1, 0], color='orange', edgecolor='black')
axes[1, 0].set_title('Distribution - Calories moyennes', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Calories moyennes')
axes[1, 0].set_ylabel('Nombre d\'utilisateurs')

# Protéines moyennes
user_profiles_filtered['avg_protein'].hist(bins=50, ax=axes[1, 1], color='purple', edgecolor='black')
axes[1, 1].set_title('Distribution - Protéines moyennes', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Protéines moyennes (%)')
axes[1, 1].set_ylabel('Nombre d\'utilisateurs')

# Préférence tags "easy"
user_profiles_filtered['pref_tag_easy'].hist(bins=30, ax=axes[1, 2], color='teal', edgecolor='black')
axes[1, 2].set_title('Distribution - Préférence "easy"', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Proportion de recettes "easy"')
axes[1, 2].set_ylabel('Nombre d\'utilisateurs')

plt.tight_layout()
plt.show()

print(f'\n✅ Visualisations terminées')

In [ ]:
# Heatmap de corrélation (features principales)
feature_cols = ['n_interactions', 'avg_rating', 'avg_minutes', 'avg_calories', 
                'avg_protein', 'pref_tag_easy', 'pref_tag_dietary']
corr_matrix = user_profiles_filtered[feature_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Heatmap - Corrélations entre Features Utilisateurs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ÉTAPE 6 - Export Final

In [ ]:
print("=" * 80)
print("EXPORT FINAL - USERS_PROFILES.CSV")
print("=" * 80)

# Export du dataset pour clustering
output_file = DATA_PROCESSED / 'users_profiles.csv'
user_profiles_filtered.to_csv(output_file, index=False)

print(f'\n✅ FICHIER EXPORTÉ:')
print(f'   • Chemin: {output_file}')
print(f'   • Shape: {user_profiles_filtered.shape}')
print(f'   • Taille: {output_file.stat().st_size / 1024 / 1024:.2f} MB')

print(f'\n📋 COLONNES EXPORTÉES ({len(user_profiles_filtered.columns)}):')
for i, col in enumerate(user_profiles_filtered.columns, 1):
    print(f'   {i:2d}. {col}')

print(f'\n✅ Dataset prêt pour K-Means Clustering!')

## 📋 Synthèse Finale

In [ ]:
print("\n" + "=" * 80)
print("🎉 SYNTHÈSE FINALE - FEATURE ENGINEERING")
print("=" * 80)

print(f'\n🎯 OBJECTIF ATTEINT:')
print(f'   Créer des profils utilisateurs agrégés pour clustering K-Means')

print(f'\n📊 DONNÉES SOURCES:')
print(f'   • Recipes clean:      {len(recipes_clean):,}')
print(f'   • Interactions clean: {len(interactions_clean):,}')

print(f'\n🔧 TRAITEMENTS RÉALISÉS:')
print(f'   1. ✅ Jointure interactions ↔ recettes (inner join)')
print(f'   2. ✅ Agrégation par user_id (moyenne pour nutrition/temps, proportion pour tags)')
print(f'   3. ✅ Filtrage utilisateurs (min {min_interactions} interactions)')
print(f'   4. ✅ Vérification qualité (0 valeurs manquantes)')

print(f'\n📦 DATASET FINAL:')
print(f'   • Fichier: users_profiles.csv')
print(f'   • Utilisateurs: {len(user_profiles_filtered):,}')
print(f'   • Features: {len(user_profiles_filtered.columns) - 1}')
print(f'   • Catégories de features:')
print(f'      - Comportement: 2 (n_interactions, avg_rating)')
print(f'      - Temps & Complexité: 3 (avg_minutes, avg_n_steps, avg_n_ingredients)')
print(f'      - Nutrition: 7 (calories, protéines, sucre, sodium, etc.)')
print(f'      - Tags Préférés: 7 (proportions time_to_make, course, dietary, etc.)')

print(f'\n🎯 PROCHAINE ÉTAPE:')
print(f'   ✅ Notebook: 04_clustering_kmeans.ipynb')
print(f'      → Normalisation des features (StandardScaler)')
print(f'      → Déterminer K optimal (elbow method, silhouette)')
print(f'      → K-Means clustering (5 segments attendus)')
print(f'      → Analyse et interprétation des clusters')
print("=" * 80)

# 🔧 Feature Engineering - Clustering Utilisateurs

## Étapes
1. ✅ Charger données parsées (recipes + interactions)
2. ✅ Gérer missing values nutrition → imputation médiane
3. ✅ Gérer outliers nutrition → winsorization 99ème percentile
4. ✅ Créer features agrégées par utilisateur (~23 features)
5. ✅ Normaliser et exporter

**Output:** `user_features_raw.csv` + `user_features_scaled.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

PROCESSED_PATH = Path('../data/processed')
DATA_PATH = Path('../data/raw')

print("✅ Imports OK")

✅ Configuration terminée


## 1️⃣ Chargement des Données

In [ ]:
# Charger les données
print("📂 Chargement...\n")

train = pd.read_csv(DATA_PATH / 'interactions_train.csv')
recipes = pd.read_csv(PROCESSED_PATH / 'recipes_enriched_full.csv')

print(f"✅ Train: {len(train):,} interactions")
print(f"✅ Recettes: {len(recipes):,} (déjà parsées)")
print(f"   Colonnes: {list(recipes.columns[:10])}...")

# Colonnes nutrition
nutrition_cols = ['calories', 'total_fat', 'sugar', 'sodium', 'protein', 'saturated_fat', 'carbohydrates']
category_cols = [c for c in recipes.columns if c.startswith('is_')]
print(f"✅ Nutrition cols: {nutrition_cols}")
print(f"✅ Category cols: {category_cols}")

📂 Chargement des données...

✅ Train: 698,901 interactions
✅ Train: 698,901 interactions
✅ Recettes enrichies: 231,637 recettes
   (nutrition + tags + temps déjà parsés)

📋 Colonnes disponibles: 19
✅ Données chargées
✅ Recettes enrichies: 231,637 recettes
   (nutrition + tags + temps déjà parsés)

📋 Colonnes disponibles: 19
✅ Données chargées


## 2️⃣ Gérer les Valeurs Manquantes Nutrition

In [ ]:
# Analyser valeurs manquantes nutrition
print("? Valeurs manquantes nutrition:\n")
missing_count = recipes[nutrition_cols].isnull().sum()
print(missing_count[missing_count > 0])
print(f"\nTotal missing: {recipes[nutrition_cols].isnull().sum().sum():,}")

# Stratégie simple:
# 1. Supprimer recettes avec >2 valeurs manquantes
# 2. Imputer le reste à la médiane

missing_per_row = recipes[nutrition_cols].isnull().sum(axis=1)
recipes_before = len(recipes)
recipes = recipes[missing_per_row <= 2].copy()
print(f"\n✅ Suppression recettes avec >2 missing: {recipes_before - len(recipes):,} recettes supprimées")

# Imputation médiane
for col in nutrition_cols:
    if recipes[col].isnull().sum() > 0:
        median = recipes[col].median()
        recipes[col].fillna(median, inplace=True)
        print(f"  {col}: imputé {recipes[col].isnull().sum()} avec {median:.1f}")

print(f"\n✅ Après traitement: {len(recipes):,} recettes, {recipes[nutrition_cols].isnull().sum().sum()} missing")

🔗 Fusion des données...

✅ Fusion terminée
   Avant: 698,901 lignes
   Après: 698,901 lignes
   Colonnes: 24 colonnes
✅ Fusion terminée
   Avant: 698,901 lignes
   Après: 698,901 lignes
   Colonnes: 24 colonnes

⚠️ Valeurs manquantes détectées:
   time_category: 2,559 (0.37%)

⚠️ Valeurs manquantes détectées:
   time_category: 2,559 (0.37%)


## 3️⃣ Gérer les Outliers Nutritionnels

In [ ]:
# Winsorisation: capping aux 99e percentile
print("⚠️ Gestion outliers nutrition (99e percentile):\n")

for col in nutrition_cols:
    p99 = recipes[col].quantile(0.99)
    p1 = recipes[col].quantile(0.01)
    count_above = (recipes[col] > p99).sum()
    count_below = (recipes[col] < p1).sum()
    
    if count_above > 0 or count_below > 0:
        print(f"{col}: {count_above} > p99({p99:.1f}), {count_below} < p1({p1:.1f})")
        recipes.loc[recipes[col] > p99, col] = p99
        recipes.loc[recipes[col] < p1, col] = p1

print(f"\n✅ Outliers cappés à [1e, 99e percentile]")

🥗 Agrégation des profils nutritionnels...

✅ Features nutritionnelles: 7 colonnes
   Utilisateurs: 25,076

📊 Aperçu:
         avg_calories  avg_total_fat  avg_sugar  avg_sodium  avg_protein  \
user_id                                                                    
1533           396.68          34.23      43.44       22.00        34.18   
1535           468.38          29.82     111.86       30.16        26.12   
1634           460.71          36.73      44.19       26.33        39.67   
1676           494.90          36.17      43.88       37.00        67.54   
1773           459.90          40.00      24.50       63.00        50.00   

         avg_saturated_fat  avg_carbohydrates  
user_id                                        
1533                 51.44              10.12  
1535                 39.32              19.55  
1634                 51.83              13.38  
1676                 36.12              11.21  
1773                 47.00               9.00  


## 4️⃣ Créer Features Agrégées par Utilisateur

In [ ]:
# Fusion interactions + recettes
print("🔗 Fusion interactions + recettes...\n")

recipes_merge = recipes.rename(columns={'id': 'recipe_id'})
data = train.merge(recipes_merge[['recipe_id'] + nutrition_cols + category_cols + ['minutes', 'time_category', 'n_tags']], 
                   on='recipe_id', how='left')

print(f"✅ Fusion: {len(data):,} lignes")
print(f"   Colonnes: {len(data.columns)}")

# Agréger par user_id
print("\n📊 Agrégation par utilisateur...\n")

# 1. Features nutritionnelles (7)
nutrition_features = data.groupby('user_id')[nutrition_cols].mean().round(2)
nutrition_features.columns = [f'avg_{c}' for c in nutrition_cols]

# 2. Features catégorielles (7) - en pourcentage
category_features = data.groupby('user_id')[category_cols].mean().round(3) * 100
category_features.columns = [f'pct_{c}' for c in category_cols]

# 3. Features temporelles (4)
time_avg = data.groupby('user_id')['minutes'].mean().round(1)
time_pcts = pd.get_dummies(data[['user_id', 'time_category']], columns=['time_category'])
time_pcts = time_pcts.groupby('user_id').mean() * 100

time_features = pd.DataFrame({
    'avg_minutes': time_avg,
    'pct_rapide': time_pcts.get('time_category_rapide', 0),
    'pct_moyen': time_pcts.get('time_category_moyen', 0),
    'pct_long': time_pcts.get('time_category_long', 0)
})

# 4. Features comportementales (5)
behavior = data.groupby('user_id').agg({
    'rating': ['count', 'mean', 'std'],
    'recipe_id': 'nunique',
    'n_tags': 'mean'
}).round(2)
behavior.columns = ['n_interactions', 'avg_rating', 'rating_std', 'n_unique_recipes', 'avg_complexity']
behavior['rating_std'] = behavior['rating_std'].fillna(0)

# Fusion toutes les features
user_features = nutrition_features.join(category_features).join(time_features).join(behavior)

print(f"✅ Features créées:")
print(f"   Utilisateurs: {len(user_features):,}")
print(f"   Features: {len(user_features.columns)}")
print(f"   Colonnes: {list(user_features.columns)}")
print(f"\n{user_features.describe().round(2).T}")

🏷️ Agrégation des préférences catégorielles...

✅ Features catégorielles: 7 colonnes
   Utilisateurs: 25,076

📊 Aperçu (en %):
         pct_is_desserts  pct_is_rapide  pct_is_sante  pct_is_plat_principal  \
user_id                                                                        
1533                13.9           80.9          42.6                   51.3   
1535                37.6           90.0          37.6                   21.7   
1634                 6.2          100.0          41.7                   56.2   
1676                 0.0           87.5          33.3                   87.5   
1773                 0.0            0.0           0.0                   50.0   

         pct_is_accompagnement  pct_is_petit_dejeuner  pct_is_occasion  
user_id                                                                 
1533                      53.0                    7.0             62.6  
1535                      24.0                   28.2             33.3  
1634                

## 5️⃣ Nettoyage & Normalisation

In [ ]:
# Nettoyage
print("🧹 Nettoyage & Filtrage...\n")

print(f"Avant: {len(user_features):,} users")

# Supprimer users avec <5 interactions
user_features = user_features[user_features['n_interactions'] >= 5].copy()
print(f"Après filtrage (<5 interactions): {len(user_features):,} users")

# Supprimer users avec trop de missing values
missing_per_user = user_features.isnull().sum(axis=1)
user_features = user_features[missing_per_user <= 3].copy()
print(f"Après filtrage (>3 missing): {len(user_features):,} users")

# Imputer valeurs manquantes restantes
for col in user_features.columns:
    if user_features[col].isnull().sum() > 0:
        user_features[col].fillna(user_features[col].median(), inplace=True)

print(f"✅ Valeurs manquantes: {user_features.isnull().sum().sum()}\n")

# Normalisation StandardScaler
print("📏 Normalisation (StandardScaler)...\n")
scaler = StandardScaler()
user_features_scaled = pd.DataFrame(
    scaler.fit_transform(user_features),
    index=user_features.index,
    columns=user_features.columns
)

print(f"✅ Mean: {user_features_scaled.mean().mean():.6f}")
print(f"✅ Std: {user_features_scaled.std().mean():.6f}")

⏱️ Agrégation des préférences temporelles...

✅ Features temporelles: 4 colonnes
   Utilisateurs: 25,076

📊 Aperçu:
         avg_minutes  pct_time_rapide  pct_time_moyen  pct_time_long
user_id                                                             
1533            92.7        34.782609       33.913043      31.304348
1535            75.6        45.762712       32.665639      21.263482
1634            35.2        70.833333       22.916667       6.250000
1676            61.1        33.333333       25.000000      29.166667
1773           200.0         0.000000        0.000000     100.000000
✅ Features temporelles: 4 colonnes
   Utilisateurs: 25,076

📊 Aperçu:
         avg_minutes  pct_time_rapide  pct_time_moyen  pct_time_long
user_id                                                             
1533            92.7        34.782609       33.913043      31.304348
1535            75.6        45.762712       32.665639      21.263482
1634            35.2        70.833333       22.916667  

## 6️⃣ Export des Fichiers

In [ ]:
# Export
print("💾 Export...\n")

# Fichier brut (pour interprétation)
raw_file = PROCESSED_PATH / 'user_features_raw.csv'
user_features.to_csv(raw_file)
print(f"✅ {raw_file.name}")
print(f"   {len(user_features):,} users × {len(user_features.columns)} features")

# Fichier normalisé (pour clustering)
scaled_file = PROCESSED_PATH / 'user_features_scaled.csv'
user_features_scaled.to_csv(scaled_file)
print(f"✅ {scaled_file.name}")

# Scaler (pour réutilisation)
import pickle
scaler_file = PROCESSED_PATH / 'scaler.pkl'
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ {scaler_file.name}")

print(f"\n📊 RÉSUMÉ FINAL:")
print(f"   ✅ Recettes nettoyées: {len(recipes):,}")
print(f"   ✅ Utilisateurs: {len(user_features):,}")
print(f"   ✅ Features: {len(user_features.columns)}")
print(f"   ✅ Fichiers: 3 (raw + scaled + scaler)")
print(f"\n🎯 Prêt pour clustering!")

📊 Agrégation des comportements...

✅ Features comportementales: 5 colonnes
   Utilisateurs: 25,076

📊 Aperçu:
         n_interactions  avg_rating  rating_std  n_unique_recipes  \
user_id                                                             
1533                115        4.75        0.72               115   
1535                649        4.48        0.78               649   
1634                 48        3.88        1.52                48   
1676                 24        4.58        0.97                24   
1773                  2        4.50        0.71                 2   

         avg_complexity  
user_id                  
1533              22.69  
1535              18.69  
1634              23.25  
1676              20.79  
1773              18.50  
